# EGC Enzyme Tree

Reads `.a.ec.txt` files from `Metagenome_Samples` and writes one `.enzyme.json` per sample.

The tree's shape comes from the EC numbers themselves (1.1.1.1 sits under 1.1.1, under 1.1, under 1). Names come from ExPASy ENZYME (SIB Swiss Institute of Bioinformatics, CC BY 4.0), saved locally by `FastAPI/scripts/download_expasy_enzyme.py`. Numbers renumbered since JGI annotated the samples follow `FastAPI/api/ec_renumbering.py`, which matches the IUBMB enzyme list.

**Run order:** Cell by cell top to bottom. Restart kernel before re-running.

In [1]:
import json
import re
import time
from pathlib import Path
from collections import defaultdict
import sys
# The repo root: the nearest folder, from where this runs upward, that
# holds DataStructures/ -- so no path below is tied to one machine.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'DataStructures').is_dir())
sys.path.append(str(REPO / 'Notebooks'))
from All_Coverages import allfold
from weighted_totals import calculate_denominator_egc
from enzyme_ec_utils import (
    build_ec_nodes, parse_enzclass, parse_enzyme_dat,
    resolve_or_drop_ec, unexplained_retired_ids,
)

## 1. Load sample list

In [2]:
with open(REPO / 'sample_names.txt') as f:
    samples = [s.strip() for s in f]
print(f"Loaded {len(samples)} samples")

Loaded 128 samples


## 2. Read enzyme annotation files → enz_db

`enz_db[gene_id]` holds three parallel lists — one entry per observation of that gene:
- `lookupId`: the EC label from column 2 of the `.a.ec.txt` file
- `timepoint`: the sample this observation came from
- `scaffolds`: the scaffold this gene sits on

Both branches of the if/else write to the same `enz_db` parameter so no observations are lost.

In [3]:
def read_enzyme(directpath, filename, s, enz_db):
    """
    Parse one sample's .a.ec.txt and append each gene observation to enz_db.
    Both branches write to the enz_db parameter — no shadowing.
    """
    path = directpath / filename
    if not path.exists():
        print(f"  WARNING: {path} not found, skipping")
        return

    with open(path, "r") as f:
        for line in f:
            parts   = line.rstrip("\n").split("\t")
            gene_id = parts[0]
            label   = parts[2]           # EC label, e.g. "EC:3.6.1.23"
            scaff   = gene_id.rsplit('_', 2)[0]

            if gene_id not in enz_db:
                enz_db[gene_id] = {
                    'lookupId':  [label],
                    'timepoint': [s],
                    'scaffolds': [scaff],
                }
            else:
                enz_db[gene_id]['lookupId'].append(label)
                enz_db[gene_id]['timepoint'].append(s)
                enz_db[gene_id]['scaffolds'].append(scaff)


base   = REPO / 'Metagenome_Samples'
enz_db = {}

for s in samples:
    read_enzyme(base / s, f"{s}.a.ec.txt", s, enz_db)

print(f"enz_db: {len(enz_db):,} genes")

enz_db: 22,293,001 genes


## 2.5 Load ExPASy ENZYME and build the tree's nodes

Reads the local copies of `enzclass.txt` (class, subclass and sub-subclass names) and `enzyme.dat` (one entry per EC number) from `DataStructures/Enzyme/`, and builds a flat `ec_nodes` dict with name, depth, category (the top-level class), parent_id and parent_name for every class and every current EC number.

ExPASy's lists sometimes run behind the IUBMB list. The two tables in `ec_renumbering.py` cover that: `TRANSFERRED_EC` (old numbers and where their genes now go) and `CURRENT_EC_NOT_IN_EXPASY` (current numbers ExPASy lacks). If JGI used a number that ExPASy lists as retired and neither table covers, this cell stops so a person can look it up.

In [4]:
def normalize_ec(ec):
    """Strip 'EC:' prefix if present, e.g. 'EC:3.6.1.23' → '3.6.1.23'."""
    return ec[3:] if ec.startswith("EC:") else ec


enzyme_dir = REPO / 'DataStructures' / 'Enzyme'
with open(enzyme_dir / 'expasy_enzyme.meta.json', encoding='utf-8') as f:
    expasy_meta = json.load(f)
classes = parse_enzclass((enzyme_dir / 'enzclass.txt').read_text(encoding='latin-1'))
entries = parse_enzyme_dat((enzyme_dir / 'enzyme.dat').read_text(encoding='latin-1'))
ec_nodes = build_ec_nodes(classes, entries)

jgi_ec_ids = {normalize_ec(label) for g in enz_db.values() for label in g['lookupId']}
unexplained = unexplained_retired_ids(jgi_ec_ids, entries)
if unexplained:
    raise ValueError(
        f"JGI used EC numbers that ExPASy lists as retired: {unexplained} -- look each "
        "one up in the IUBMB list (enzyme-database.org) and add it to TRANSFERRED_EC "
        "or CURRENT_EC_NOT_IN_EXPASY in FastAPI/api/ec_renumbering.py"
    )

print(f"ExPASy ENZYME release {expasy_meta['release']}: {len(classes)} classes, "
      f"{len(ec_nodes) - len(classes):,} current EC numbers")
for k in sorted((k for k, v in ec_nodes.items() if v['depth'] == 1), key=int):
    print(f"  {k}: {ec_nodes[k]['name']}")

ExPASy ENZYME release 04-Nov-2026: 410 classes, 7,012 current EC numbers
  1: Oxidoreductases
  2: Transferases
  3: Hydrolases
  4: Lyases
  5: Isomerases
  6: Ligases
  7: Translocases


## 3. Build enzyme tree

For each gene in `enz_db`, strip the `EC:` prefix and decide which node it counts under (`resolve_or_drop_ec`): a renumbered EC goes to its new number; a current EC to itself; a partial EC (e.g. `1.1.1.-`) or one in no list to its nearest real class.
Walk up the `parent_id` chain to the root, registering the gene and scaffold at **every ancestor node**.
Only nodes with at least one observation from `enz_db` are added to the tree.

In [5]:
enz_tree = {}  # {ec_id: {name, category, depth, parent_id, parent_name, timepoints}}
dropped_ec_ids   = set()  # ids with no resolvable node at any level
resolved_partial_ec_ids = set()  # ids credited somewhere other than their own node (renumbered, partial, or walked up) (resolve_or_drop_ec, FastAPI/api/ec_renumbering.py)

start = time.time()

for gene_id, ginfo in enz_db.items():
    lookups    = ginfo.get('lookupId',  [])
    timepoints = ginfo.get('timepoint', [])
    scaffolds  = ginfo.get('scaffolds', [])

    for ec_raw, sample, scaff in zip(lookups, timepoints, scaffolds):
        # Assert exactly one id per entry instead of
        # silently taking only the first of a ';'-joined list -- checked
        # against real data, this never currently fires (every entry has
        # exactly one id), so if it ever does, that's a real format change
        # worth failing loudly on rather than silently dropping ids.
        segments = ec_raw.split(';')
        assert len(segments) == 1, (
            f"Expected exactly one EC id per lookupId entry, got "
            f"{len(segments)} in {ec_raw!r} (gene {gene_id}) -- "
            f"the raw .a.ec.txt format may have changed."
        )
        ec_id = normalize_ec(segments[0].strip())

        # resolve_or_drop_ec handles every case in one tested
        # implementation -- a renumbered id, credited to its new place in
        # the tree (e.g. DNA helicase "3.6.4.12" -> "5.6.2", see
        # TRANSFERRED_EC); an id that's already a real node; a
        # partial/wildcard id (e.g. "1.1.1.-"); or a fully-specified id in
        # no list (e.g. "7.2.4.8"), which walks up to its nearest real class.
        start_id = resolve_or_drop_ec(ec_id, ec_nodes)
        if start_id is None:
            # Nothing resolves at any level. Dropped, but listed below
            # rather than silently (see docs/MISSING_ANNOTATIONS.md).
            dropped_ec_ids.add(ec_id)
            continue
        if start_id != ec_id:
            resolved_partial_ec_ids.add(ec_id)

        # Walk from the deepest REAL node (start_id -- the EC itself when
        # fully resolved, or its deepest resolved ancestor for a partial
        # id) up to the root, registering at every ancestor
        current = start_id
        while current is not None:
            if current not in ec_nodes:
                break

            if current not in enz_tree:
                b = ec_nodes[current]
                enz_tree[current] = {
                    'name':        b['name'],
                    'category':    b['category'],
                    'depth':       b['depth'],
                    'parent_id':   b['parent_id'],
                    'parent_name': b['parent_name'],
                    'timepoints':  {},
                }

            node = enz_tree[current]

            if sample not in node['timepoints']:
                node['timepoints'][sample] = {
                    'egc':       0,
                    'norm_egc':  0.0,
                    'scaffolds': {},
                    'genes':     set(),
                }

            tp = node['timepoints'][sample]
            # Only count this gene once per (EC node, sample) — a multi-domain
            # protein can produce multiple rows for the same EC in .a.ec.txt.
            if gene_id not in tp['genes']:
                tp['scaffolds'][scaff] = tp['scaffolds'].get(scaff, 0) + 1
            tp['genes'].add(gene_id)

            current = ec_nodes[current]['parent_id']

print(f"Enzyme tree: {len(enz_tree):,} nodes  ({time.time() - start:.1f}s)")
print(f"Resolved {len(resolved_partial_ec_ids)} distinct EC ids credited to another node (renumbered, partial, or walked up)")
print(f"Dropped {len(dropped_ec_ids)} distinct EC ids with no resolvable node at any level (see docs/MISSING_ANNOTATIONS.md)")
if dropped_ec_ids:
    print(f"  e.g. {sorted(dropped_ec_ids)[:10]}")

Enzyme tree: 3,585 nodes  (101.6s)
Resolved 155 distinct EC ids credited to another node (renumbered, partial, or walked up)
Dropped 0 distinct EC ids with no resolvable node at any level (see docs/MISSING_ANNOTATIONS.md)


## 4. Compute EGC and norm_egc

`egc = Σ (gene_count_on_scaffold × scaffold_coverage)`, using only genes annotated with an EC number (from this notebook's own `enz_db`).

`norm_egc = egc / weighted_totals[sample]`, where the denominator comes from `Notebooks/weighted_totals.py` (`calculate_denominator_egc`) and is the SAME value used by `EGC_KEGG_TREE.ipynb` / `EGC_COG_TREE.ipynb` / `EGC_TAX_TREE.ipynb` for a given sample -- computed once from every gene with any annotation across all four sources (via the mapping TSVs), not just genes with an EC hit. That shared scope is what makes norm_egc comparable across annotation types on the same plot. norm_egc still stays in (0, 1], since `egc` (an Enzyme-only subset of the shared total) can never exceed it.

In [6]:
# Cross-source-comparable denominator.
#
# calculate_denominator_egc() (Notebooks/weighted_totals.py) computes ONE
# shared total per sample from the mapping TSVs -- every gene annotated by
# ANY of the four sources (taxonomy/KEGG/enzyme/COG), counted once each --
# so this notebook uses the exact same denominator as EGC_KEGG_TREE.ipynb /
# EGC_COG_TREE.ipynb / EGC_TAX_TREE.ipynb for a given sample. That's what
# makes norm_egc comparable across annotation types on the same plot,
# instead of each notebook normalizing against only its own source's
# annotated genes.
weighted_totals = calculate_denominator_egc(samples)

# Sanity check on first sample: compare against the OLD unweighted,
# not-gene-count-weighted sum (allfold[sample]['total']) as a rough
# magnitude check -- not a correctness check on its own.
_s0 = next(iter(weighted_totals))
_new = weighted_totals[_s0]
_old = allfold.get(_s0, {}).get('total', 0)
print(f'Sample {_s0}: old denom={_old:,.0f}  new denom={_new:,.0f}')
print(f'norm_egc values will be ~{_old/_new:.2f}x smaller, now bounded by 1.0')

Sample 3300060358: old denom=8,298,642  new denom=16,137,978
norm_egc values will be ~0.51x smaller, now bounded by 1.0


In [7]:
def avg_gene_map(node_id):
    node = enz_tree[node_id]
    for sample_name, data in node['timepoints'].items():
        scaffold_counts = data.get('scaffolds')
        if not scaffold_counts or sample_name not in allfold:
            continue

        covs = allfold[sample_name]
        sample_total_depth = weighted_totals[sample_name]

        total_egc = 0
        for scaff, cnt in scaffold_counts.items():
            cov = covs.get(scaff, 0)
            total_egc += cnt * cov

        data['egc']      = total_egc
        data['norm_egc'] = (
            total_egc / sample_total_depth
            if sample_total_depth > 0 else 0.0
        )


for node_id in enz_tree:
    avg_gene_map(node_id)
print('EGC computation complete')


EGC computation complete


## 5. Write .enzyme.json files

In [8]:
base = REPO / 'DataStructures' / 'Enzyme'
base.mkdir(parents=True, exist_ok=True)

for s in samples:
    tp_out = {}
    for ec_id, node in enz_tree.items():
        if s not in node['timepoints']:
            continue
        tp = node['timepoints'][s]
        tp_out[ec_id] = {
            'name':        node['name'],
            'depth':       node['depth'],
            'category':    node['category'],
            'parent_id':   node['parent_id'],
            'parent_name': node['parent_name'],
            'egc':         tp.get('egc',       0),
            'norm_egc':    tp.get('norm_egc',  0.0),
            'scaffolds':   tp.get('scaffolds', {}),
        }
    with open(base / f"{s}.enzyme.json", 'w') as f:
        json.dump(tp_out, f, indent=2)

print(f"Wrote {len(samples)} .enzyme.json files to {base}")

Wrote 128 .enzyme.json files to c:\Users\sakoehn\Desktop\JGI\DataStructures\Enzyme
